# Phase 7: Retriever

**Project:** Hospital Patient Helpdesk Chatbot  
**Purpose:** Retrieve the most relevant, citable hospital document chunks for a user question.

This notebook imports `06_rag_pipeline/07_retriever.py`, which remains the single source of truth for query embedding, routing, reranking, confidence, safety labels, evaluation, and outputs.

## Input and output contract

**Inputs:** `05_vector_store/chroma_db/06_vector_index.sqlite3` and `01_data/sample_queries/test_questions.csv`.

**Data outputs:** `07_retrieval_results.json`, `07_retrieval_report.json`, `07_retrieval_audit.csv`, and `07_failed_queries.json`.

**Plots:** `plots/07_top_retrieval_score_distribution.png` and `plots/07_retrieval_latency_by_query.png`.

## Retrieval method

The retriever embeds the question with the same Phase 5 model used by the index, applies an explicit metadata route for unambiguous topics, requests cosine candidates from Phase 6, and reranks them with a weighted blend of normalized vector similarity and query-token coverage. It returns evidence only; answer generation belongs to Phase 10 and medical safety enforcement belongs to Phase 11.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Locate the project from the workspace, project, or notebooks folder."""
    start = start.resolve()
    candidates = []
    for directory in (start, *start.parents):
        candidates.extend((directory, directory / "hospital_patient_helpdesk_chatbot"))
    for candidate in candidates:
        module = candidate / "06_rag_pipeline" / "07_retriever.py"
        index = candidate / "05_vector_store" / "chroma_db" / "06_vector_index.sqlite3"
        if module.is_file() and index.is_file():
            return candidate
    raise FileNotFoundError("Could not locate the Hospital Patient Helpdesk Chatbot project.")

PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_PATH = PROJECT_ROOT / "06_rag_pipeline" / "07_retriever.py"
INDEX_PATH = PROJECT_ROOT / "05_vector_store" / "chroma_db" / "06_vector_index.sqlite3"
QUESTIONS_PATH = PROJECT_ROOT / "01_data" / "sample_queries" / "test_questions.csv"
OUTPUT_DIR = PROJECT_ROOT / "01_data" / "processed"
spec = importlib.util.spec_from_file_location("phase7_retriever", MODULE_PATH)
retriever = importlib.util.module_from_spec(spec)
assert spec and spec.loader
sys.modules[spec.name] = retriever
spec.loader.exec_module(retriever)
embedder, vector_store = retriever.load_dependencies(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Vector index: {INDEX_PATH}")

## 1. Validate the index contract

The retriever checks SQLite integrity and reads the stored model and dimension. Query vectors are rejected if their embedding model or dimension would not match the index.

In [ ]:
contract = retriever.load_index_contract(INDEX_PATH)
print(json.dumps(contract, indent=2))
assert contract["model"] == embedder.MODEL_NAME
assert int(contract["dimension"]) == 384

## 2. Retrieve context for one hospital question

This example shows ranked chunks, source citations, hybrid scores, confidence, inferred filters, and safety-routing labels before any LLM is called.

In [ ]:
config = retriever.RetrieverConfig(top_k=5, candidate_count=30, vector_weight=0.80)
response = retriever.retrieve(
    "How can I reschedule my hospital appointment?",
    INDEX_PATH, config, embedder, vector_store,
)
print(json.dumps({
    "question": response.question,
    "confidence": response.confidence,
    "filters": response.filters,
    "safety_labels": response.safety_labels,
    "results": [
        {"rank": r.rank, "source": r.source_file, "score": r.final_score, "department": r.department}
        for r in response.results
    ],
}, indent=2))

## 3. Apply an explicit metadata filter

Callers can constrain retrieval to an approved department or content category. Explicit filters override inferred category routing and are returned in the response for auditability.

In [ ]:
filtered = retriever.retrieve(
    "Where is the department?", INDEX_PATH, config, embedder, vector_store,
    department="Cardiology", content_category="department_information",
)
assert all(result.department == "Cardiology" for result in filtered.results)
print([(result.source_file, result.final_score) for result in filtered.results])

## 4. Inspect safety-routing labels

Phase 7 labels emergency and unsafe-medical-advice wording while still retrieving evidence. It does not diagnose or prescribe; Phase 11 uses these labels to enforce the final response policy.

In [ ]:
emergency = retriever.retrieve(
    "I have severe chest pain. What is wrong with me?",
    INDEX_PATH, config, embedder, vector_store,
)
assert "emergency" in emergency.safety_labels
assert "unsafe_medical_advice" in emergency.safety_labels
print(f"Safety labels: {emergency.safety_labels}")
print(f"Top evidence: {emergency.results[0].source_file}")

## 5. Evaluate the bundled test questions

The shared module evaluates all questions, checks whether the expected source appears in the top-k evidence, records latency and confidence, writes failures separately, and generates diagnostics.

In [ ]:
evaluation = retriever.run_retriever_evaluation(
    INDEX_PATH, QUESTIONS_PATH, OUTPUT_DIR, config, PROJECT_ROOT
)
report = json.loads(evaluation.report_path.read_text(encoding="utf-8"))
failed = json.loads(evaluation.failed_path.read_text(encoding="utf-8"))
assert report["top_k_source_hits"] == report["questions"] == 12
assert failed == []
print(json.dumps(report, indent=2))

## 6. Display diagnostic plots

The score histogram shows retrieval strength across the test set. The latency chart helps identify unexpectedly slow questions or storage regressions.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(evaluation.score_plot_path)))
display(Image(filename=str(evaluation.latency_plot_path)))

## Notebook and Python module differences

- The notebook offers guided contract checks, single-query inspection, explicit filtering, safety-label examples, assertions, and inline plots.
- The Python module owns dependency loading, compatible query embedding, category routing, hybrid reranking, confidence, safety labels, evaluation, artifacts, plots, and CLI behavior.
- The notebook imports the production module rather than duplicating retrieval logic.

## Next step

Use the ranked `RetrievalResponse` in `08_prompt_template.py` to construct a grounded healthcare-support prompt with citations and safety instructions.